# AeroPulse — Maintenance Auto Loader Audit Integration

## Purpose

Integrate the reusable AeroPulse audit framework with
the Maintenance Auto Loader Bronze ingestion pipeline.

## Objectives

1. Generate a unique pipeline run identifier.
2. Record pipeline start time.
3. Execute Auto Loader ingestion.
4. Capture ingestion metrics.
5. Record SUCCESS or FAILED status.
6. Capture exceptions when ingestion fails.
7. Record pipeline completion time.
8. Reuse the existing centralized audit framework.

## Architecture

Maintenance Application
        ↓
Raw Landing Volume
        ↓
Auto Loader
        ↓
Bronze Maintenance
        ↓
Audit Framework

## Design Principle

Audit logic must remain reusable and independent from
individual source-specific ingestion logic.

In [0]:
from datetime import datetime, timezone
import uuid

from pyspark.sql import functions as F

In [0]:
ENVIRONMENT = "DEV"

SOURCE_SYSTEM = "maintenance_app"
SOURCE_ENTITY = "maintenance"

BRONZE_TABLE = (
    "workspace.aeropulse_dev.bronze_maintenance"
)

SOURCE_PATH = (
    "/Volumes/workspace/aeropulse_dev/"
    "raw_landing/maintenance_app/maintenance"
)

CHECKPOINT_PATH = (
    "/Volumes/workspace/aeropulse_dev/"
    "raw_landing/_checkpoints/"
    "maintenance_app/maintenance"
)

SCHEMA_LOCATION = (
    "/Volumes/workspace/aeropulse_dev/"
    "raw_landing/_schemas/"
    "maintenance_app/maintenance"
)

In [0]:
pipeline_run_id = str(uuid.uuid4())

print("Pipeline Run ID:", pipeline_run_id)

In [0]:
pipeline_start_time = datetime.now(timezone.utc)

print(
    "Pipeline Start Time:",
    pipeline_start_time
)

In [0]:
import sys
sys.path.append("/Workspace/Users/hclearningtools08@gmail.com/aeropulse-databricks-lakehouse/src")

from audit.pipeline_audit import start_pipeline_run

start_pipeline_run(
    spark=spark,
    audit_table="workspace.aeropulse_dev.pipeline_audit",
    pipeline_run_id=pipeline_run_id,
    pipeline_name="maintenance_autoloader",
    environment=ENVIRONMENT,
    layer="BRONZE",
    source_system=SOURCE_SYSTEM,
    target_table=BRONZE_TABLE
)

In [0]:
display(
    spark.table("workspace.aeropulse_dev.pipeline_audit")
    .filter(
        F.col("pipeline_run_id") == pipeline_run_id
    )
)

In [0]:
maintenance_stream_df = (
    spark.readStream
    .format("cloudFiles")
    .option(
        "cloudFiles.format",
        "json"
    )
    .option(
        "cloudFiles.schemaLocation",
        SCHEMA_LOCATION
    )
    .load(SOURCE_PATH)
)

In [0]:
maintenance_bronze_df = (
    maintenance_stream_df
    .withColumn(
        "_pipeline_run_id",
        F.lit(pipeline_run_id)
    )
    .withColumn(
        "_ingestion_timestamp",
        F.current_timestamp()
    )
    .withColumn(
        "_source_system",
        F.lit(SOURCE_SYSTEM)
    )
    .withColumn(
        "_source_entity",
        F.lit(SOURCE_ENTITY)
    )
    .withColumn(
        "_source_file_path",
        F.col("_metadata.file_path")
    )
)

In [0]:
maintenance_query = (
    maintenance_bronze_df
    .writeStream
    .format("delta")
    .option(
        "checkpointLocation",
        CHECKPOINT_PATH
    )
    .trigger(
        availableNow=True
    )
    .toTable(
        BRONZE_TABLE
    )
)

In [0]:
maintenance_query.awaitTermination()

In [0]:
print(
    "Query status:",
    maintenance_query.status
)

In [0]:
print(
    "Query active:",
    maintenance_query.isActive
)

In [0]:
bronze_count = (
    spark.table(BRONZE_TABLE)
    .count()
)

print(
    "Bronze record count:",
    bronze_count
)

In [0]:
progress = maintenance_query.lastProgress

print(progress)

In [0]:
records_processed = 0

if progress:
    records_processed = (
        progress.get("numInputRows", 0)
    )

print(
    "Records processed:",
    records_processed
)

In [0]:
pipeline_end_time = datetime.now(timezone.utc)

print(
    "Pipeline End Time:",
    pipeline_end_time
)

In [0]:
pipeline_duration_seconds = (
    pipeline_end_time - pipeline_start_time
).total_seconds()

print(
    "Pipeline Duration (seconds):",
    pipeline_duration_seconds
)

In [0]:
display(
    spark.table("workspace.aeropulse_dev.pipeline_audit")
    .filter(
        F.col("pipeline_run_id") == pipeline_run_id
    )
)